# Listing Supply Intelligence

*Given what we know about **demand** and **supply** in a region, should we prioritize bringing more properties of a given type onto the platform?*

## Setup

In [1]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
provider = os.getenv("LLM_PROVIDER", "openai")
if provider == "vocareum":
    client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=os.getenv("VOCAREUM_API_KEY"))
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Load Ciudaty Data

In [4]:
demand = pd.read_csv("demand_signals.csv")

In [5]:
print("Demand signals:")
display(demand)

Demand signals:


,region,property_type,bedrooms,search_volume,lead_count,recent_sales,demand_score
0,west,apartment,2,1240,86,34,0.91
1,north,apartment,1,620,31,12,0.54
2,south,house,3,710,42,18,0.68
3,east,apartment,2,980,73,21,0.82
4,center,studio,1,430,19,8,0.47


In [6]:
supply = pd.read_csv("supply_snapshot.csv")

In [7]:
print("\nSupply snapshot:")
display(supply)


Supply snapshot:


,region,property_type,bedrooms,active_listings,avg_days_on_market,supply_score
0,west,apartment,2,18,21,0.22
1,north,apartment,1,77,54,0.81
2,south,house,3,36,39,0.55
3,east,apartment,2,29,28,0.37
4,center,studio,1,91,63,0.88


## RealThor v0.1

The system prompt is defined inline here

In [8]:
SYSTEM_PROMPT = """
You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile.

Use the available data to recommend whether the company should prioritize bringing more
properties of that type to the platform.

Your recommendation must include:
- priority: high, medium, or low
- reasoning based on demand
- reasoning based on supply
- recommended action
- confidence: high, medium, or low

Do not make claims that are not supported by the provided data.
"""

In [9]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [10]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

## First Recommendation

In [11]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)
print(recommendation)

- **Priority:** High
- **Reasoning based on demand:** The demand signals indicate a strong interest in 2-bedroom apartments in the west region, with a search volume of 1,240 and a lead count of 86. This suggests that many potential buyers are actively looking for this type of property, and the high demand score of 0.91 further emphasizes the strong interest.
- **Reasoning based on supply:** The supply snapshot shows that there are only 18 active listings available for 2-bedroom apartments in the west region, which is notably low relative to the high demand. The average days on market is 21, indicating that properties are selling relatively quickly, thus further highlighting the supply constraint with a low supply score of 0.22.
- **Recommended action:** Prioritize bringing more 2-bedroom apartments to the platform in the west region to align supply with the strong demand.
- **Confidence:** High


## Try Another Segment

In [12]:
recommendation = ask_realthor(region="center", property_type="apartment", bedrooms=1)
print(recommendation)

Priority: Low  
Reasoning based on demand: Without any available data for the requested segment, it is impossible to assess the current demand for properties in this category.  
Reasoning based on supply: Similarly, the lack of data indicates that there may also be uncertainties in existing supply for these properties.  
Recommended action: It is advisable to gather more data about the specific segment before making any decisions regarding prioritization of property listings.  
Confidence: High


---

## What we just built

RealThor works. But:

- The system prompt is **hardcoded inline**. If we change it, we lose the previous version.
- There is **no project structure** — the notebook and the data are in the same flat folder.
- There is **no environment pinning** — `pip install openai` could install any version.
- If a teammate runs this notebook tomorrow and gets a different answer, **we cannot explain why**.

We fix all of this soon.